# FIFA World Cup LLM Fine-tuning with DPO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Section 1: Setup and Environment Initialization

In [ ]:
# This section sets up the Google Colab environment.
# It includes mounting Google Drive to access files, changing the working directory,
# installing necessary libraries, and checking the available GPU resources for training.

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# %cd /content/drive/MyDrive/CSC4182-202425/Lab Session4

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes evaluate
!pip install -q rouge_score bert_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00


In [ ]:
import torch
print("GPU available :", torch.cuda.is_available())
print("GPU name      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("VRAM          :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

GPU available : True
GPU name      : Tesla T4
VRAM          : 15.6 GB


In [ ]:
import pandas as pd, random, re
random.seed(42)

WC_PATH       = "/content/drive/MyDrive/LLM_fine-tuning(Qwen)/WorldCups.csv"
MATCHES_PATH  = "/content/drive/MyDrive/LLM_fine-tuning(Qwen)/WorldCupMatches.csv"
wc      = pd.read_csv(WC_PATH)
matches = pd.read_csv(MATCHES_PATH)

## Section 2: Data Loading and Preference Record Generation

In [ ]:
# This section loads the FIFA World Cup data from CSV files into pandas DataFrames.
# It then proceeds to generate 'preference records' which are crucial for DPO training.
# These records consist of a 'prompt', a 'chosen' response (correct/preferred), and a 'rejected' response (incorrect/less preferred),
# covering various scenarios like winner/host questions, match results, and style preferences (e.g., rejecting verbose answers).

In [ ]:
all_countries = sorted(set(wc["Winner"]) | set(wc["Country"]) | set(wc["Runners-Up"]))

def wrong_country(true_country):
    choices = [c for c in all_countries if c != true_country]
    return random.choice(choices)

def prompt_text(question: str) -> str:
    # Same chat template as the SFT stage, truncated before the assistant turn
    return (
        f"<|system|>You are a FIFA World Cup expert assistant.</s>\n"
        f"<|user|>{question.strip()}</s>\n"
        f"<|assistant|>"
    )

def completion(answer: str) -> str:
    return f"{answer.strip()}</s>"

pref_records = []

# 1. Winner / host questions — corrupt the country, keep everything else identical
for _, r in wc.iterrows():
    yr = int(r["Year"])

    q = f"Who won the {yr} FIFA World Cup?"
    chosen   = f"{r['Winner']} won the {yr} FIFA World Cup, which was hosted by {r['Country']}."
    rejected = f"{wrong_country(r['Winner'])} won the {yr} FIFA World Cup, which was hosted by {r['Country']}."
    pref_records.append({"prompt": prompt_text(q), "chosen": completion(chosen), "rejected": completion(rejected)})

    q = f"Which country hosted the {yr} FIFA World Cup?"
    chosen   = f"The {yr} FIFA World Cup was hosted by {r['Country']}."
    rejected = f"The {yr} FIFA World Cup was hosted by {wrong_country(r['Country'])}."
    pref_records.append({"prompt": prompt_text(q), "chosen": completion(chosen), "rejected": completion(rejected)})

    q = f"Who was the runner-up at the {yr} FIFA World Cup?"
    chosen   = f"The runner-up at the {yr} FIFA World Cup was {r['Runners-Up']}."
    rejected = f"The runner-up at the {yr} FIFA World Cup was {wrong_country(r['Runners-Up'])}."
    pref_records.append({"prompt": prompt_text(q), "chosen": completion(chosen), "rejected": completion(rejected)})

# 2. Match-result questions — corrupt the scoreline
matches_clean = matches.dropna(subset=["Home Team Name", "Away Team Name", "Home Team Goals", "Away Team Goals"])
for _, r in matches_clean.sample(min(400, len(matches_clean)), random_state=42).iterrows():
    home, away = str(r["Home Team Name"]).strip(), str(r["Away Team Name"]).strip()
    hg, ag = int(r["Home Team Goals"]), int(r["Away Team Goals"])
    yr, stage = int(r["Year"]), str(r["Stage"]).strip()

    if hg > ag:   true_result = f"{home} beat {away} {hg}\u2013{ag}"
    elif ag > hg: true_result = f"{away} beat {home} {ag}\u2013{hg}"
    else:         true_result = f"{home} and {away} drew {hg}\u2013{ag}"

    # corrupt: flip / inflate the scoreline so it's wrong but still plausible-looking
    fake_hg, fake_ag = hg + random.choice([1, 2]), ag
    if fake_hg > fake_ag:   fake_result = f"{home} beat {away} {fake_hg}\u2013{fake_ag}"
    elif fake_ag > fake_hg: fake_result = f"{away} beat {home} {fake_ag}\u2013{fake_hg}"
    else:                   fake_result = f"{home} and {away} drew {fake_hg}\u2013{fake_ag}"

    q = f"What was the result of the {home} vs {away} match at the {yr} FIFA World Cup?"
    chosen   = f"In the {stage} stage of the {yr} FIFA World Cup, {true_result}."
    rejected = f"In the {stage} stage of the {yr} FIFA World Cup, {fake_result}."
    pref_records.append({"prompt": prompt_text(q), "chosen": completion(chosen), "rejected": completion(rejected)})

# 3. Style preference — same correct fact, but reject overly verbose/hedgy phrasing
hedge_templates = [
    "I think it might have been {ans}, but I'm not 100% certain.",
    "As far as I can recall, possibly {ans}, though it could be wrong.",
    "It's hard to say for sure, but I believe the answer is roughly {ans}.",
]
for _, r in wc.sample(min(40, len(wc)), random_state=7).iterrows():
    yr = int(r["Year"])
    q = f"Who won the {yr} FIFA World Cup?"
    chosen   = f"{r['Winner']} won the {yr} FIFA World Cup, which was hosted by {r['Country']}."
    hedge    = random.choice(hedge_templates).format(ans=f"{r['Winner']}")
    pref_records.append({"prompt": prompt_text(q), "chosen": completion(chosen), "rejected": completion(hedge)})

random.shuffle(pref_records)
print(f"Total preference pairs generated: {len(pref_records)}")
print("\n\u2500\u2500 Sample pair \u2500\u2500")
print("PROMPT  :", pref_records[0]["prompt"])
print("CHOSEN  :", pref_records[0]["chosen"])
print("REJECTED:", pref_records[0]["rejected"])


Total preference pairs generated: 480

── Sample pair ──
PROMPT  : <|system|>You are a FIFA World Cup expert assistant.</s>
<|user|>What was the result of the Hungary vs Mexico match at the 1958 FIFA World Cup?</s>
<|assistant|>
CHOSEN  : In the Group 3 stage of the 1958 FIFA World Cup, Hungary beat Mexico 4–0.</s>
REJECTED: In the Group 3 stage of the 1958 FIFA World Cup, Hungary beat Mexico 6–0.</s>


In [ ]:

from datasets import Dataset

n = len(pref_records)
train_end = int(0.80 * n)
val_end   = int(0.90 * n)

dpo_train = Dataset.from_list(pref_records[:train_end])
dpo_val   = Dataset.from_list(pref_records[train_end:val_end])
dpo_test  = Dataset.from_list(pref_records[val_end:])

print(f"Train : {len(dpo_train)}")
print(f"Val   : {len(dpo_val)}")
print(f"Test  : {len(dpo_test)}")

Train : 384
Val   : 48
Test  : 48


## Section 3: Dataset Splitting

In [ ]:
# This section takes the generated preference records and splits them into training,
# validation, and test datasets. This separation is essential for evaluating the model's
# performance on unseen data and preventing overfitting during training.

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_ADAPTER_PATH = "/content/drive/MyDrive/DPO/qwen25-worldcup-final-updated/Fine-tuned Model"


## Section 4: Model Configuration and Loading

In [ ]:
# Here, the base pre-trained language model (Qwen 2.5 1.5B Instruct) and its tokenizer are loaded.
# Quantization settings (BitsAndBytesConfig) are applied to reduce memory footprint and speed up computation.
# The SFT (Supervised Fine-Tuning) adapter, previously fine-tuned, is attached and kept trainable
# for further alignment with DPO (Direct Preference Optimization).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,    #  float32 compute, no AMP (same as SFT stage)
    bnb_4bit_use_double_quant=True,
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_PATH, trust_remote_code=True)
tokenizer.pad_token        = tokenizer.eos_token
tokenizer.padding_side     = "right"
tokenizer.model_max_length = 512

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False

# Attach the SFT LoRA adapter, kept trainable for DPO
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH, is_trainable=True)
model.print_trainable_parameters()

print("Parameter dtypes:", set(p.dtype for p in model.parameters()))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Parameter dtypes: {torch.bfloat16, torch.float32, torch.uint8}


In [ ]:
# Sanity check: how many optimizer steps will we actually get?
per_device_bs = 1
grad_accum = 4   # match whatever you set below in DPOConfig
epochs = 2

total_steps = (len(dpo_train) // (per_device_bs * grad_accum)) * epochs
print("Training examples:", len(dpo_train))
print("Expected optimizer steps:", total_steps)

Training examples: 384
Expected optimizer steps: 192


In [ ]:
from trl import DPOTrainer, DPOConfig

DPO_OUTPUT_DIR = "/content/drive/MyDrive/DPO"

dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    beta=0.1,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=10,
    logging_steps=2,
    learning_rate=5e-6,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=10,
    load_best_model_at_end=True,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    processing_class=tokenizer,
)

dpo_trainer.train()


DPO_SAVE_PATH = "/content/drive/MyDrive/DPO/qwen25-worldcup-dpo-final"
dpo_trainer.model.save_pretrained(DPO_SAVE_PATH)
tokenizer.save_pretrained(DPO_SAVE_PATH)
print("DPO-aligned model saved to:", DPO_SAVE_PATH)



Adding EOS to train dataset:   0%|          | 0/384 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/384 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
10,0.687988,0.686605,0.181338,5965.000000,-3.362946,-3.205168,0.934424,0.000034,-0.012411,0.604167,0.012447,-4.013133,-8.871745
20,0.681152,0.678630,0.180814,12084.000000,-3.375681,-3.218542,0.933708,-0.003762,-0.033742,0.750000,0.029963,-4.051076,-9.084961
30,0.682129,0.669759,0.179654,18079.000000,-3.386391,-3.234161,0.932292,-0.007974,-0.057245,0.812500,0.049291,-4.093363,-9.320312
40,0.676270,0.663005,0.178200,24098.000000,-3.398841,-3.246942,0.932251,-0.015221,-0.078934,0.812500,0.063690,-4.165848,-9.537109
50,0.668945,0.658529,0.176514,30081.000000,-3.408086,-3.259904,0.929091,-0.023276,-0.097242,0.833333,0.073931,-4.246043,-9.720703
60,0.638672,0.652262,0.174908,36038.000000,-3.416913,-3.272252,0.929123,-0.029567,-0.118979,0.812500,0.089447,-4.309489,-9.937174
70,0.656738,0.647502,0.173426,42132.000000,-3.426936,-3.282139,0.926958,-0.035476,-0.135547,0.833333,0.100094,-4.367966,-10.103190
80,0.611084,0.642456,0.171343,48086.000000,-3.436035,-3.295096,0.928181,-0.047511,-0.160075,0.812500,0.112484,-4.488673,-10.348958
90,0.601074,0.638468,0.170139,54001.000000,-3.440729,-3.303466,0.928427,-0.055350,-0.179788,0.854167,0.124344,-4.566996,-10.545573
100,0.632812,0.635620,0.168671,60049.000000,-3.450075,-3.313127,0.929093,-0.060240,-0.192707,0.875000,0.132482,-4.616384,-10.673828


## Section 5: DPO Trainer Setup and Training

In [ ]:
# This section configures the DPO training parameters, including learning rate, batch sizes,
# number of epochs, and optimization strategy. The `DPOTrainer` from the `trl` library is initialized
# with the model, datasets, and configurations. Finally, the training process is initiated,
# and the DPO-aligned model and tokenizer are saved to a specified path.

In [ ]:
import matplotlib.pyplot as plt

log_history = dpo_trainer.state.log_history
train_steps  = [x["step"] for x in log_history if "loss" in x and "eval_loss" not in x]
train_losses = [x["loss"] for x in log_history if "loss" in x and "eval_loss" not in x]
val_steps    = [x["step"] for x in log_history if "eval_loss" in x]
val_losses   = [x["eval_loss"] for x in log_history if "eval_loss" in x]

reward_steps = [x["step"] for x in log_history if "rewards/margins" in x]
margins      = [x["rewards/margins"] for x in log_history if "rewards/margins" in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_steps, train_losses, label="Train Loss", color="steelblue", linewidth=2)
axes[0].plot(val_steps, val_losses, label="Validation Loss", color="coral", linewidth=2, linestyle="--")
axes[0].set_xlabel("Step"); axes[0].set_ylabel("DPO Loss"); axes[0].set_title("DPO Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

if margins:
    axes[1].plot(reward_steps, margins, label="Reward Margin (chosen - rejected)", color="seagreen", linewidth=2)
    axes[1].axhline(0, color="gray", linestyle=":")
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("Margin"); axes[1].set_title("Reward Margin (higher = better separation)")
    axes[1].legend(); axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "No reward margin logged", ha="center")

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/CSC4182-202425/Lab Session4/dpo_training_plot.png", dpi=150)
plt.show()
print("Plot saved!")



## Section 6: Training Visualization

In [ ]:
# This section extracts training metrics (loss and reward margins) from the DPO trainer's log history.
# It then generates and displays plots to visualize the training and validation loss over steps,
# as well as the reward margin, which indicates how well the model is learning to differentiate
# between chosen and rejected responses. The plot is also saved as an image file.

In [ ]:
model.eval()

test_questions = [
    "Who won the 1970 FIFA World Cup?",
    "Which country has won the most FIFA World Cups?",
    "Who was the runner-up at the 2014 FIFA World Cup?",
]

def generate(q, use_adapter=True):
    prompt = (
        f"<|system|>You are a FIFA World Cup expert assistant.</s>\n"
        f"<|user|>{q}</s>\n"
        f"<|assistant|>"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    ctx = model.disable_adapter() if not use_adapter else torch.no_grad()
    with torch.no_grad():
        if not use_adapter:
            with model.disable_adapter():
                outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        else:
            outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for question in test_questions:
    print(f"\nQ: {question}")
    print(f"A (DPO-aligned): {generate(question, use_adapter=True)}")



## Section 7: Model Evaluation (Initial Test)

In [ ]:
# This section performs an initial qualitative evaluation of the DPO-aligned model.
# It defines a helper `generate` function to get model responses for given questions.
# A small set of test questions is used to demonstrate the model's ability to answer FIFA World Cup related queries
# after DPO alignment, providing a quick check of its performance.

In [ ]:
model.eval()

test_questions = [
    "Who won the 1970 FIFA World Cup?",
    "Which country has won the most FIFA World Cups?",
    "Who was the runner-up at the 2014 FIFA World Cup?",
    "Who won the 1998 FIFA World Cup?",
    "Which country hosted the 2002 FIFA World Cup?",
    "How many times has Brazil won the FIFA World Cup?",
    "Who was the runner-up at the 1990 FIFA World Cup?",
    "Which country hosted the first FIFA World Cup?",
    "Who won the FIFA World Cup in 2006?",
    "Which European country has won the most World Cups?",
]

def generate(q, use_adapter=True):
    prompt = (
        f"<|system|>You are a FIFA World Cup expert assistant.</s>\n"
        f"<|user|>{q}</s>\n"
        f"<|assistant|>"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        if not use_adapter:
            with model.disable_adapter():
                outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        else:
            outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for question in test_questions:
    print(f"\nQ: {question}")
    print(f"A (SFT only):    {generate(question, use_adapter=False)}")
    print(f"A (DPO-aligned): {generate(question, use_adapter=True)}")

## Section 8: Model Evaluation (Extended Comparison)

In [ ]:
# In this final section, a more extensive evaluation is conducted.
# The model's responses are generated for a larger set of test questions,
# comparing the output from the SFT-only model (base model without DPO adapter enabled)
# and the DPO-aligned model. This comparison highlights the impact of the DPO training
# on the model's ability to generate preferred and accurate answers.